# Plant Disease Classification using Deep Learning

## Project Overview

The objective of this project is to develop a deep learning-based image classification system capable of automatically identifying plant diseases from leaf images. The project investigates the effectiveness of both custom convolutional neural networks and transfer learning approaches for large-scale plant disease recognition.

The study is conducted using the **New Plant Diseases Dataset (Augmented)**, which contains images of healthy and diseased plant leaves across 38 categories. Multiple model architectures are evaluated and compared in terms of classification performance, computational efficiency, and model complexity.

---

The project follows a progressive experimental pipeline:

1. Dataset Overview and Exploratory Data Analysis (EDA)
2. Data Pipeline Development
3. Baseline CNN Training and Evaluation
4. MobileNetV2 Transfer Learning
5. EfficientNetV2B0 Transfer Learning
6. Model Comparison
7. Final Conclusions

---

## Table of Contents

- [Phase 1: Dataset Overview and Exploratory Data Analysis](#phase-1-dataset-overview-and-exploratory-data-analysis-eda)
- [Phase 2: Data Pipeline Development](#phase-2-data-pipeline-development)
- [Phase 3: Baseline CNN Training and Evaluation](#phase-3-baseline-cnn-training-and-evaluation)
- [Phase 4: MobileNetV2 Transfer Learning](#phase-4-mobilenetv2-transfer-learning)

---

# Phase 1: Dataset Overview and Exploratory Data Analysis (EDA)

## 1.1 Dataset Description

This project uses the **New Plant Diseases Dataset (Augmented)** for multi-class plant disease classification. The dataset contains images of healthy and diseased plant leaves collected under controlled conditions and organized into separate class folders.

The dataset structure consists of:

* Training set (`train`)
* Validation set (`valid`)
* Test set (`test`)

Each class corresponds to a specific plant species and disease condition.

---

## 1.2 Dataset Statistics

| Metric                   | Value           |
| ------------------------ | --------------- |
| Number of Classes        | 38              |
| Total Training Images    | 70,295          |
| Largest Class Size       | 2,022 images    |
| Smallest Class Size      | 1,642 images    |
| Average Images per Class | 1,849.87 images |

---

## 1.3 Class Distribution Analysis

The dataset is highly balanced across all 38 classes.

* Largest class: **Soybean___healthy** (2,022 images)
* Smallest class: **Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot** (1,642 images)

The ratio between the largest and smallest classes is approximately:

2022 / 1642 ≈ 1.23

This indicates very low class imbalance, reducing the risk of model bias toward specific classes.

Therefore, additional balancing techniques such as oversampling, undersampling, or class weighting are not expected to be necessary during training.

---

## 1.4 Image Resolution Analysis

All sampled images have identical dimensions:

* Width: 256 pixels
* Height: 256 pixels

The uniform image size simplifies preprocessing and ensures consistency across the dataset.

Dataset image resolution:

256 × 256 × 3 (RGB)

---

## 1.5 Visual Inspection

Random samples from multiple classes were examined.

Observations:

1. Leaf regions occupy most of the image area.
2. Backgrounds are relatively simple and consistent.
3. Disease symptoms such as spots, lesions, discoloration, and mildew are visually distinguishable.
4. Images appear to have undergone augmentation and orientation variation.
5. Image quality is generally high with clear focus on leaf structures.

These characteristics make the dataset well suited for convolutional neural networks and transfer learning approaches.

---

## 1.6 Implications for Model Development

Based on the EDA findings:

* No severe class imbalance exists.
* No image resizing inconsistencies were detected.
* Disease patterns are visually observable.
* Dataset quality is high.
* Transfer learning models such as MobileNetV2 or EfficientNetB0 are expected to perform well.

Consequently, the next stage of the project will focus on building an efficient image classification pipeline using TensorFlow and transfer learning techniques.

## 1.7 Phase Summary

Based on the exploratory analysis, the dataset was found to be highly suitable for supervised image classification. The class distribution is well balanced, image quality is consistently high, and disease symptoms are visually distinguishable. These characteristics suggest that both custom CNNs and transfer learning models should perform effectively on this task.

---


# Phase 2: Data Pipeline Development

## 2.1 Objectives

The objective of Phase 2 was to build an efficient and scalable data pipeline for training deep learning models on the New Plant Diseases Dataset. This phase focused on dataset loading, preprocessing, data augmentation, and input pipeline optimization.

---

## 2.2 Data Loading and Preprocessing

### Dataset Loading

The dataset was loaded using TensorFlow's `image_dataset_from_directory()` utility. This approach automatically scans the directory structure, assigns labels based on folder names, and generates TensorFlow datasets suitable for model training.

Configuration:

```text
Image Size : 256 × 256
Batch Size : 32
Number of Classes : 38
Label Mode : Categorical (One-Hot Encoding)
```

Training and validation datasets were loaded separately from their respective directories.

---

### Label Encoding

Class labels were automatically inferred from folder names and converted into one-hot encoded vectors.

Example:

```text
Apple___Apple_scab
        ↓
[1, 0, 0, ..., 0]

Apple___Black_rot
        ↓
[0, 1, 0, ..., 0]
```

The resulting label tensor shape was:

```text
(batch_size, 38)
```

which is compatible with multi-class classification using categorical cross-entropy loss.

---

### Dataset Verification

Several validation checks were performed to ensure the pipeline was functioning correctly.

Observed batch shapes:

```text
Images Shape : (32, 256, 256, 3)
Labels Shape : (32, 38)
```

These results confirm that:

* Images are correctly loaded as RGB images.
* All images are resized consistently to 256×256.
* Labels are encoded correctly for the 38-class classification task.

Additionally, random batches were visualized to verify that image-label mappings were correct.

---

### Data Augmentation

To improve model generalization while preserving disease-related visual characteristics, a lightweight augmentation strategy was adopted.

Applied augmentations:

* Random Horizontal Flip
* Random Rotation (±3%)

The final augmentation pipeline was intentionally conservative because:

* The dataset already contains augmented samples.
* Disease symptoms often depend on color and texture patterns.
* Excessive transformations may distort lesions, spots, and discoloration that are important for classification.

---

## 2.3 Input Pipeline Optimization

### Prefetching

TensorFlow's `prefetch()` operation was applied to both training and validation datasets.

```python
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
valid_ds = valid_ds.prefetch(AUTOTUNE)
```

Prefetching allows data preparation and model execution to occur concurrently, reducing GPU idle time during training.

Benefits:

* Improved training throughput
* Better GPU utilization
* Reduced data loading bottlenecks

---

### Automatic Performance Tuning

The pipeline uses:

```python
tf.data.AUTOTUNE
```

which enables TensorFlow to automatically determine the optimal level of parallelism and buffering based on available hardware resources.

---

### Dataset Statistics After Pipeline Construction

| Dataset        | Number of Batches |
| -------------- | ----------------: |
| Training Set   |             2,197 |
| Validation Set |               550 |

Pipeline configuration:

```text
Image Size : 256 × 256
Batch Size : 32
Classes : 38
Training Batches : 2,197
Validation Batches : 550
```

---

## 2.4 Final Pipeline Workflow

```text
Dataset Directories
        ↓
image_dataset_from_directory()
        ↓
Label Encoding
        ↓
Data Augmentation
        ↓
Batching
        ↓
Prefetch (AUTOTUNE)
        ↓
Ready for Model Training
```

---

## 2.5 Phase Summary

A complete and optimized TensorFlow data pipeline was successfully established. The resulting pipeline provides efficient data loading, augmentation, batching, and GPU utilization, forming the foundation for all subsequent experiments.

---

# Phase 3: Baseline CNN Training and Evaluation

## 3.1 Objectives

The objective of Phase 3 was to establish a reference performance benchmark using a custom
convolutional neural network trained from scratch. This baseline serves as the comparison
point for subsequent transfer learning models.

---

## 3.2 Model Architecture

The baseline model was designed as a sequential CNN with four convolutional blocks followed
by a classification head.

### Architecture Summary

| Component        | Details                                |
| ---------------- | -------------------------------------- |
| Input            | 256 × 256 × 3                          |
| Normalization    | Rescaling (1/255)                      |
| Conv Block 1     | Conv2D(32) + BN + ReLU + MaxPool       |
| Conv Block 2     | Conv2D(64) + BN + ReLU + MaxPool       |
| Conv Block 3     | Conv2D(128) + BN + ReLU + MaxPool      |
| Conv Block 4     | Conv2D(256) + BN + ReLU + MaxPool      |
| Pooling          | Global Average Pooling                 |
| Dense Head       | Dense(256) + BN + ReLU + Dropout(0.3) |
| Output           | Dense(38) + Softmax                    |
| Total Parameters | 466,918                                |

Each convolutional block applies Batch Normalization before activation to stabilize training
and accelerate convergence. Global Average Pooling replaces Flatten to reduce parameter
count and improve generalization. A Dropout rate of 0.3 was applied before the output layer
to further reduce overfitting.

---

## 3.3 Training Configuration

### Optimizer and Loss Function

| Setting           | Value                   |
| ----------------- | ----------------------- |
| Optimizer         | Adam                    |
| Learning Rate     | 1e-3                    |
| Loss Function     | Categorical Crossentropy|
| Epochs            | 20                      |
| Batch Size        | 32                      |

### Training Callbacks

Three callbacks were used during training:

* **ModelCheckpoint** — saves the best model weights based on `val_accuracy`.
* **EarlyStopping** — halts training if `val_loss` does not improve for 5 consecutive
  epochs, and restores the best weights automatically.
* **ReduceLROnPlateau** — reduces the learning rate by a factor of 0.2 if `val_loss`
  does not improve for 2 consecutive epochs.

This configuration ensures training stops before overfitting occurs and that the saved
model always corresponds to the best observed validation performance.

---

## 3.4 Evaluation Results

### Overall Performance

| Metric             | Value   |
| ------------------ | ------- |
| Best Epoch         | 19 / 20 |
| Train Accuracy     | 99.74%  |
| Validation Accuracy| 99.78%  |
| Train Loss         | 0.0091  |
| Validation Loss    | 0.0075  |
| Precision (Macro)  | 99.78%  |
| Recall (Macro)     | 99.78%  |
| F1-Score (Macro)   | 99.78%  |
| Total Parameters   | 466,918 |

The model achieved 99.78% validation accuracy at epoch 19, with no signs of overfitting
as both training and validation losses decreased consistently throughout training.

### Per-Class Performance

Classification was evaluated across all 38 classes using precision, recall, and F1-score.
The majority of classes achieved perfect scores of 1.00 across all three metrics. Minor
deviations were observed in a small number of classes:

| Class                                              | Precision | Recall | F1   |
| -------------------------------------------------- | --------- | ------ | ---- |
| Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot | 0.99      | 0.98   | 0.99 |
| Corn_(maize)___Northern_Leaf_Blight                | 0.98      | 0.99   | 0.99 |
| Tomato___Bacterial_spot                            | 1.00      | 0.99   | 0.99 |
| Tomato___Target_Spot                               | 1.00      | 0.98   | 0.99 |
| Tomato___Late_blight                               | 0.99      | 1.00   | 1.00 |

These minor misclassifications are consistent with the visual similarity between certain
disease symptoms, particularly among Corn and Tomato disease categories.

---

## 3.5 Discussion

### Dataset Characteristics

The high validation accuracy is consistent with results reported in existing literature
on this dataset. Several characteristics of the New Plant Diseases Dataset (Augmented)
contribute to this outcome:

* Images were collected under controlled conditions with uniform backgrounds.
* Disease symptoms are clearly visible and occupy most of the image area.
* The validation set is derived from the same augmentation pipeline as the training set,
  which may reduce the effective difficulty of the evaluation.

It should be noted that the provided test set contains only 33 images with labels encoded
in filenames using inconsistent formatting, making systematic evaluation infeasible.
Therefore, the validation set (17,572 images across 38 classes) is used as the primary
evaluation set for all models in this project.

### Baseline Significance

Despite the high absolute accuracy, the baseline CNN serves an important role in this
project. The key comparison across models is not accuracy alone, but the trade-off between:

* Model complexity (number of parameters)
* Convergence speed (epochs required)
* Computational efficiency (training and inference time)

The baseline CNN, with only 466,918 parameters and no pretrained weights, achieving
99.78% validation accuracy demonstrates that this dataset does not require a large
or complex model. Transfer learning models in subsequent phases will be evaluated
against this reference point.

---

## 3.6 Saved Artifacts

A lightweight custom CNN trained from scratch successfully achieved 99.78% validation
accuracy on the 38-class plant disease classification task. The trained model and all
evaluation artifacts were saved for downstream comparison.

**Saved Artifacts**

| Artifact                          | Location                              |
| --------------------------------- | ------------------------------------- |
| Trained model                     | `models/baseline_cnn.keras`           |
| Metrics summary                   | `reports/baseline_metrics.json`       |
| Accuracy learning curve           | `figures/baseline_accuracy.png`       |
| Loss learning curve               | `figures/baseline_loss.png`           |
| Confusion matrix                  | `figures/baseline_confusion_matrix.png` |

## 3.7 Phase Summary

The baseline CNN achieved 99.78% validation accuracy while using fewer than 500k parameters. This result establishes a strong benchmark for evaluating transfer learning approaches in later phases.

---

# Phase 4: MobileNetV2 Transfer Learning

## 4.1 Objectives

The objective of Phase 4 was to apply transfer learning using MobileNetV2 pretrained on
ImageNet, and evaluate whether pretrained feature extractors provide any advantage over
the custom baseline CNN on this dataset.

---

## 4.2 Model Architecture

MobileNetV2 was used as the feature extractor with a custom classification head appended
on top. The base model was initialized with ImageNet weights and the top layers were
excluded.

### Architecture Summary

| Component        | Details                                |
| ---------------- | -------------------------------------- |
| Input            | 256 × 256 × 3                          |
| Normalization    | Rescaling (1/255)                      |
| Augmentation     | Random Horizontal Flip + Rotation (±3%)|
| Base Model       | MobileNetV2 (ImageNet, exclude top)    |
| Base Layers      | 155 layers                             |
| Pooling          | Global Average Pooling                 |
| Dense Head       | Dense(256) + BN + ReLU + Dropout(0.3) |
| Output           | Dense(38) + Softmax                    |
| Total Parameters | 2,596,710                              |

The same augmentation strategy from Phase 2 was retained to ensure consistency across
all models and enable fair comparison.

---

## 4.3 Training Strategy

A two-phase training strategy was adopted to stabilize learning and maximize the benefit
of pretrained weights.

### Phase 1: Head Training

| Setting           | Value                    |
| ----------------- | ------------------------ |
| Base Model        | Frozen (155 layers)      |
| Optimizer         | Adam                     |
| Learning Rate     | 1e-3                     |
| Max Epochs        | 10                       |
| Early Stopping    | patience = 3 (val_loss)  |

In Phase 1, the entire MobileNetV2 base was frozen and only the classification head was
trained. This allows the head to adapt to the target domain before any pretrained weights
are modified.

### Phase 2: Fine-Tuning

| Setting           | Value                    |
| ----------------- | ------------------------ |
| Frozen Layers     | 115 (bottom layers)      |
| Unfrozen Layers   | 40 (top layers)          |
| Optimizer         | Adam                     |
| Learning Rate     | 1e-4                     |
| Max Epochs        | 15                       |
| Early Stopping    | patience = 5 (val_loss)  |
| ReduceLROnPlateau | factor = 0.2, patience=2 |

In Phase 2, the top 40 layers of the base model were unfrozen and trained jointly with
the classification head using a reduced learning rate of 1e-4 to avoid disrupting the
pretrained feature representations.

### Training Callbacks

* **ModelCheckpoint** — saves the best model weights based on `val_accuracy`.
* **EarlyStopping** — halts training if `val_loss` does not improve within the patience
  window and restores the best weights automatically.
* **ReduceLROnPlateau** — applied during fine-tuning to further reduce the learning rate
  when validation loss plateaus (Phase 2 only).

---

## 4.4 Evaluation Results

### Overall Performance

| Metric              | Value      |
| ------------------- | ---------- |
| Best Epoch          | 23 (total) |
| Train Accuracy      | 99.93%     |
| Validation Accuracy | 99.60%     |
| Train Loss          | 0.0024     |
| Validation Loss     | 0.0155     |
| Precision (Macro)   | 99.58%     |
| Recall (Macro)      | 99.58%     |
| F1-Score (Macro)    | 99.58%     |
| Total Parameters    | 2,596,710  |

The model reached its best validation accuracy at epoch 23 across both training phases
combined. A small gap between train accuracy (99.93%) and validation accuracy (99.60%)
is observed, indicating minor overfitting during fine-tuning, which is expected given
the dataset characteristics discussed in Phase 3.

### Per-Class Performance

Classification performance across all 38 classes was evaluated using precision, recall,
and F1-score. The majority of classes achieved scores of 1.00. Minor deviations were
concentrated in visually similar disease categories, consistent with Phase 3 observations.

---

## 4.5 Discussion

### Comparison with Baseline CNN

| Metric              | Baseline CNN | MobileNetV2  |
| ------------------- | ------------ | ------------ |
| Total Parameters    | 466,918      | 2,596,710    |
| Best Epoch          | 19           | 23 (2-phase) |
| Train Accuracy      | 99.74%       | 99.93%       |
| Validation Accuracy | 99.78%       | 99.60%       |
| Validation Loss     | 0.0075       | 0.0155       |
| F1-Score (Macro)    | 99.78%       | 99.58%       |

MobileNetV2 with 2,596,710 parameters — approximately 5.6× more than the baseline CNN —
did not outperform the baseline on this dataset. The baseline CNN achieved higher
validation accuracy (99.78% vs 99.60%) and lower validation loss (0.0075 vs 0.0155)
while using significantly fewer parameters.

### Why Transfer Learning Did Not Provide a Clear Advantage

This outcome is consistent with the dataset characteristics identified in Phase 1:

* Images were collected under controlled conditions with simple, uniform backgrounds.
* Disease symptoms are visually distinct and occupy most of the image area.
* The dataset is already augmented and balanced across 38 classes.

Under these conditions, the pretrained ImageNet features — which encode general
visual patterns from natural images — may not offer additional discriminative power
beyond what a well-designed CNN trained from scratch can learn directly from the data.
The fine-tuning phase introduced additional complexity without yielding accuracy gains,
suggesting the dataset does not benefit strongly from deep pretrained representations.

### Role of MobileNetV2 in the Comparison

Despite not outperforming the baseline, MobileNetV2 still serves an important role in
this project. Its result confirms that the strong baseline performance is not an artifact
of the custom architecture, but reflects the inherent characteristics of the dataset
itself. This finding motivates the evaluation of EfficientNetV2B0 in Phase 5, which
employs a more modern and efficient architecture that may better leverage pretrained
representations.

---

## 4.6 Saved Artifacts

MobileNetV2 transfer learning with two-phase training achieved 99.60% validation accuracy,
slightly below the baseline CNN (99.78%). This result suggests that for well-structured,
visually consistent datasets such as the New Plant Diseases Dataset, lightweight custom
CNNs can match or exceed the performance of significantly larger pretrained models.

**Saved Artifacts**

| Artifact              | Location                               |
| --------------------- | -------------------------------------- |
| Trained model         | `models/mobilenet_v2.keras`            |
| Metrics summary       | `reports/mobilenet_metrics.json`       |
| Accuracy learning curve | `figures/mobilenet_accuracy.png`     |
| Loss learning curve   | `figures/mobilenet_loss.png`           |
| Confusion matrix      | `figures/mobilenet_confusion_matrix.png` |

## 4.7 Phase Summary

MobileNetV2 achieved 99.60% validation accuracy but did not surpass the baseline CNN. The results suggest that on highly structured and visually consistent datasets, lightweight custom CNNs can perform competitively with significantly larger pretrained architectures.